# 04 — Document Type Classification

## Purpose
Identifies the document type and physical boundaries of each distinct
document within a parsed file. A single uploaded file may contain multiple
document types across its pages — this notebook detects those boundaries
and assigns a stable `CHILD_DOC_ID` to each segment, which becomes the
key all downstream steps (extraction, confidence scoring, review) operate
on.

## What this notebook does
Reads all pages from `DOCUMENTS_PAGES` for files with `STATUS = 'PARSED'`
that have not yet been classified, groups them by `DOC_ID`, and sends all
pages of each document together in a single `AI_COMPLETE` call. The model
returns a segment map identifying which pages belong to which document type
and where each distinct physical document begins and ends.

The classify model and prompt are loaded from `PIPELINE_CONFIG` at runtime
— no notebook edits are needed to change the model or tune the prompt.

Each segment in the response is assigned a `CHILD_DOC_ID`. For files
containing only one document type, `CHILD_DOC_ID` equals `DOC_ID`
— no split occurs. For files containing multiple document types or multiple
distinct documents of the same type, each segment gets a new UUID as its
`CHILD_DOC_ID`, with `DOC_ID` linking back to the original upload
for full lineage traceability.

`DOCUMENTS_PAGES` is then updated to tag each page with the `CHILD_DOC_ID`
of the segment it belongs to, so downstream steps can query pages by
segment without joining through `DOCUMENTS_CLASSIFIED`.

Status is updated in `DOCUMENTS_INGESTED` to `CLASSIFIED` on success or
`CLASSIFY_ERROR` on failure. The notebook is idempotent — already-classified
documents are excluded.

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_CLASSIFIED` | One row per segment — `CHILD_DOC_ID`, `DOC_ID`, `DOC_TYPE`, page range, confidence, boundary signal |
| `PROCESSING.DOCUMENTS_PAGES` | `CHILD_DOC_ID` updated per page to reflect its segment assignment |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `CLASSIFIED` or `CLASSIFY_ERROR` |

## Key design decisions
- **One AI_COMPLETE call per file, not per page** — the model sees all
  pages together so it has full context when detecting boundaries between
  adjacent documents of the same type
- **Boundary signal captured** — the model returns a brief reason for each
  new segment boundary, stored in `BOUNDARY_SIGNAL` for reviewer visibility
  and debugging
- **`CHILD_DOC_ID` is the downstream key** — all steps after classification
  operate on `CHILD_DOC_ID`, not `DOC_ID`. This cleanly separates the
  physical file identity (`DOC_ID`) from the logical document identity
  (`CHILD_DOC_ID`)
- **Config-driven** — model and prompt loaded from `PIPELINE_CONFIG` at
  runtime; no code change needed to swap models or tune the prompt

In [ ]:
import json
import re
import uuid
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from snowflake.snowpark.context import get_active_session

# Constants
DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'
CONFIG_SCHEMA     = 'CONFIG'
MAX_WORKERS = 4

s = get_active_session()

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")


In [ ]:
# Load classify config from PIPELINE_CONFIG
config = {
    row['CONFIG_KEY']: row['CONFIG_VALUE']
    for row in s.sql(f"""
        SELECT CONFIG_KEY, CONFIG_VALUE
        FROM {DB}.{CONFIG_SCHEMA}.PIPELINE_CONFIG
        WHERE CONFIG_KEY IN ('classify_model', 'classify_prompt')
    """).collect()
}

CLASSIFY_MODEL  = config.get('classify_model')
CLASSIFY_PROMPT = config.get('classify_prompt')

if not CLASSIFY_MODEL or not CLASSIFY_PROMPT:
    raise ValueError(
        "Missing classify config in PIPELINE_CONFIG — "
        "run 00_setup_config.ipynb first"
    )

info(f"Classify model : {CLASSIFY_MODEL}")
info(f"Classify prompt: {len(CLASSIFY_PROMPT)} chars")

In [ ]:
# Build classify queue
# One entry per DOC_ID with all its pages
# Only documents that are PARSED and not yet in DOCUMENTS_CLASSIFIED

rows = s.sql(f"""
    SELECT
        p.DOC_ID,
        p.PAGE_NUMBER,
        COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT) AS PAGE_CONTENT_EN
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON p.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON p.DOC_ID = c.DOC_ID
    WHERE i.STATUS = 'PARSED'
      AND c.DOC_ID IS NULL    -- not yet classified
    ORDER BY p.DOC_ID, p.PAGE_NUMBER
""").collect()

# Group pages by DOC_ID
pages_by_doc = defaultdict(list)
for row in rows:
    pages_by_doc[row['DOC_ID']].append({
        'page_number':    row['PAGE_NUMBER'],
        'page_content_en': row['PAGE_CONTENT_EN'],
    })

classify_queue = [
    {'doc_id': doc_id, 'pages': pages}
    for doc_id, pages in pages_by_doc.items()
]

info(f"Classify queue: {len(classify_queue)} document(s)")

if not classify_queue:
    print("\nNothing to classify.")

In [ ]:
def build_page_block(pages):
    return '\n\n'.join(
        f"[PAGE {p['page_number']}]\n{p['page_content_en']}"
        for p in pages
        if p['page_content_en'] and p['page_content_en'].strip()
    )

def parse_ai_response(raw):
    stripped = raw.strip()
    if stripped.startswith('"') and stripped.endswith('"'):
        stripped = json.loads(stripped)

    # Remove markdown code fences
    if '```' in stripped:
        parts   = stripped.split('```')
        content = parts[1]
        if content.startswith('json'):
            content = content[4:]
        stripped = content.strip()

    result = json.loads(stripped)

    if isinstance(result, str):
        result = json.loads(result)

    if not isinstance(result, dict):
        raise ValueError(f"Expected dict, got {type(result).__name__}")

    return result


In [ ]:
def classify_one(item):
    """
    Classifies a single document's pages.
    Returns (doc_id, segments, error_str).
    segments is a list of dicts with doc_type, language, page_start,
    page_end, confidence.
    """
    doc_id = item['doc_id']
    pages  = item['pages']

    try:
        page_block = build_page_block(pages)

        if not page_block.strip():
            raise ValueError("No parseable page content to classify")

        prompt = CLASSIFY_PROMPT.format(pages=page_block)

        raw = s.sql(
            f"SELECT AI_COMPLETE('{CLASSIFY_MODEL}', ?) AS response",
            params=[prompt]
        ).collect()[0]['RESPONSE']

        result   = parse_ai_response(raw)
        segments = result.get('segments', [])

        if not segments:
            raise ValueError("AI_COMPLETE returned empty segments array")

        return doc_id, segments, None

    except Exception as e:
        return doc_id, None, str(e)


classify_results = []   # (doc_id, segments)
classify_errors  = []

if classify_queue:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(classify_one, item): item
            for item in classify_queue
        }
        for future in as_completed(futures):
            item   = futures[future]
            doc_id = item['doc_id']
            result_doc_id, segments, err = future.result()

            if err:
                classify_errors.append({'doc_id': doc_id, 'error': err})
                error(f"  [FAIL] {doc_id}: {err}")
            else:
                classify_results.append({
                    'doc_id':   result_doc_id,
                    'segments': segments,
                })
                seg_summary = ', '.join(
                    f"{s['doc_type']} pp.{s['page_start']}-{s['page_end']}"
                    for s in segments
                )
                info(f"  [OK]   {doc_id} — {seg_summary}")


In [ ]:
# Build DOCUMENTS_CLASSIFIED rows and DOCUMENTS_PAGES updates
# Assign CHILD_DOC_ID per segment
# Single-segment doc: CHILD_DOC_ID = DOC_ID (no split)
# Multi-segment doc:  CHILD_DOC_ID = new UUID per segment

classified_rows = []
page_updates    = []   # (child_doc_id, doc_id, page_start, page_end)

for result in classify_results:
    doc_id   = result['doc_id']
    segments = result['segments']

    for seg in segments:
        # Single segment — no split, reuse parent DOC_ID as child
        if len(segments) == 1:
            child_doc_id = doc_id
        else:
            child_doc_id = str(uuid.uuid4())

        classified_rows.append({
            'CHILD_DOC_ID':   child_doc_id,
            'DOC_ID':  doc_id,
            'DOC_TYPE':       seg.get('doc_type'),
            'PAGE_START':     seg.get('page_start'),
            'PAGE_END':       seg.get('page_end'),
            'CONFIDENCE':     seg.get('confidence'),
            'BOUNDARY_SIGNAL': seg.get('boundary_signal'),
        })

        # Track which pages belong to this child segment
        page_updates.append({
            'child_doc_id': child_doc_id,
            'doc_id':       doc_id,
            'page_start':   seg.get('page_start'),
            'page_end':     seg.get('page_end'),
        })


# Write DOCUMENTS_CLASSIFIED
if classified_rows:
    s.write_pandas(
        pd.DataFrame(classified_rows),
        table_name='DOCUMENTS_CLASSIFIED',
        database=DB,
        schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(classified_rows)} row(s) to DOCUMENTS_CLASSIFIED")


# Update CHILD_DOC_ID on DOCUMENTS_PAGES
# Assign each page its segment's CHILD_DOC_ID

pages_updated = 0
for upd in page_updates:
    s.sql(f"""
        UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
        SET CHILD_DOC_ID = '{upd['child_doc_id']}'
        WHERE DOC_ID      = '{upd['doc_id']}'
          AND PAGE_NUMBER BETWEEN {upd['page_start']} AND {upd['page_end']}
    """).collect()
    pages_updated += 1

info(f"Updated CHILD_DOC_ID on {pages_updated} segment(s) in DOCUMENTS_PAGES")


#Update STATUS in DOCUMENTS_INGESTED

if classify_results:
    classified_ids = [r['doc_id'] for r in classify_results]
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'CLASSIFIED'
        WHERE DOC_ID IN ({','.join(f"'{d}'" for d in classified_ids)})
    """).collect()

if classify_errors:
    error_ids = [e['doc_id'] for e in classify_errors]
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'CLASSIFY_ERROR'
        WHERE DOC_ID IN ({','.join(f"'{d}'" for d in error_ids)})
    """).collect()
